# Gold Mart Creation and Validation

This notebook creates and validates the final Gold dimensional model for the NYC Green Taxi project.

The model follows a star schema designed for trip-level analysis and business reporting.

## Purpose

The purpose of this notebook is to transform the trusted Silver-layer data into reusable Gold dimensions and a trip-level fact table.

The final model will support analysis of trip dates, trip times, taxi zones, weather conditions, revenue, distance, duration, and passenger behavior.

## Approved Gold Tables

The final Gold layer will contain the following tables:

- `dim_date`
- `dim_time`
- `dim_taxi_zone`
- `dim_weather_hour`
- `fact_green_taxi_trip`

The fact table grain is one row per Green Taxi trip.

## Silver Source Inspection

The Gold layer will be created from the following Silver source tables:

- `green_taxi`
- `taxi_zones`
- `weather`

Before creating the Gold tables, we inspect the source structures and sample records to confirm the available columns and data types.

In [0]:
SHOW TABLES IN `ftw-week-08`.`02_silver`;

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.green_taxi;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.green_taxi
LIMIT 10;

## Green Taxi Table Structure

The `green_taxi` Silver table contains the trip-level source data.

It provides the pickup and dropoff timestamps, pickup and dropoff location identifiers, trip distance, trip duration, passenger count, and financial measures such as fare, tip, tolls, and total amount.

This table will become the main source for `fact_green_taxi_trip`.

## Green Taxi Column Inventory

The Green Taxi columns were reviewed to identify the fields needed for the fact table.

The important fields include the trip timestamps, location identifiers, trip measures, financial measures, and `batch_id` for lineage.

The source does not provide a guaranteed unique trip identifier, so a deterministic `trip_key` will be generated from stable trip attributes.

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.taxi_zones;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.taxi_zones
LIMIT 10;

## Taxi Zone Table Structure

The `taxi_zones` table provides the descriptive information for each taxi location identifier.

This table will be transformed into `dim_taxi_zone`, which will be used for both pickup-zone and dropoff-zone analysis.

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.weather;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.weather
ORDER BY weather_datetime
LIMIT 10;

## Weather Column Inventory and Preview

The `weather` table contains hourly weather observations identified by `weather_datetime`.

The available weather attributes include temperature, precipitation, rain, snowfall, weather code, and wind speed.

The weather dimension will be joined to trips using the pickup time rounded to the matching weather hour.

## Final Gold Dimensional Model

The approved star schema contains four dimensions and one fact table.

The fact table stores one row per Green Taxi trip. The dimensions provide reusable descriptive attributes for dates, times, taxi zones, and hourly weather.

Pickup and dropoff dates, times, and zones will be treated as role-playing dimensions in the fact table.

## Gold Layer Update

Completed the Gold mart transformations in Databricks using the approved star-schema design:

- Created `dim_date`
- Created `dim_time`
- Created `dim_taxi_zone`
- Created `dim_weather_hour`
- Created `fact_green_taxi_trip`

Validation passed:

- 133,367 Silver rows retained in Gold
- 133,367 unique deterministic trip keys
- 0 duplicate trip keys
- 0 row-count difference between Silver and Gold
- 0 missing date, time, or taxi-zone foreign keys
- 0 missing weather foreign keys after mapping unavailable weather to the documented Unknown member (`weather_hour_key = 0`)
- 0 duplicate rows after dimension joins

No Gold-side deduplication was applied because the previously tested five-column candidate identity produced different measures within all 384 duplicate groups.

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_date
USING DELTA
AS
WITH required_dates AS (

    SELECT TO_DATE(lpep_pickup_datetime) AS full_date
    FROM `ftw-week-08`.`02_silver`.green_taxi

    UNION

    SELECT TO_DATE(lpep_dropoff_datetime) AS full_date
    FROM `ftw-week-08`.`02_silver`.green_taxi

),

date_range AS (
    SELECT
        MIN(full_date) AS min_date,
        MAX(full_date) AS max_date
    FROM required_dates
    WHERE full_date IS NOT NULL
),

dates AS (
    SELECT EXPLODE(
        SEQUENCE(min_date, max_date, INTERVAL 1 DAY)
    ) AS full_date
    FROM date_range
)

SELECT
    CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT) AS date_key,
    full_date,
    YEAR(full_date) AS year,
    QUARTER(full_date) AS quarter,
    MONTH(full_date) AS month_number,
    DATE_FORMAT(full_date, 'MMMM') AS month_name,
    DAYOFMONTH(full_date) AS day_of_month,
    DATE_FORMAT(full_date, 'EEEE') AS day_name,
    DAYOFWEEK(full_date) AS day_of_week,
    DAYOFWEEK(full_date) IN (1, 7) AS is_weekend,
    CAST(NULL AS BOOLEAN) AS is_holiday
FROM dates;

## Create DIM_DATE

The `dim_date` table was created from the minimum and maximum Green Taxi pickup dates.

A complete calendar row was generated for every date in that range. Each date has a unique `date_key` and descriptive calendar attributes such as year, quarter, month, day, weekend indicator, and holiday indicator.

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT date_key) AS unique_date_keys,
    MIN(full_date) AS minimum_date,
    MAX(full_date) AS maximum_date,

    DATEDIFF(MAX(full_date), MIN(full_date)) + 1
        AS expected_row_count,

    COUNT(*) - (
        DATEDIFF(MAX(full_date), MIN(full_date)) + 1
    ) AS missing_date_count,

    SUM(
        CASE
            WHEN date_key != CAST(DATE_FORMAT(full_date, 'yyyyMMdd') AS INT)
            THEN 1
            ELSE 0
        END
    ) AS invalid_date_key_count,

    SUM(
        CASE
            WHEN year != YEAR(full_date)
              OR quarter != QUARTER(full_date)
              OR month_number != MONTH(full_date)
              OR day_of_month != DAYOFMONTH(full_date)
              OR day_of_week != DAYOFWEEK(full_date)
              OR is_weekend != (DAYOFWEEK(full_date) IN (1, 7))
            THEN 1
            ELSE 0
        END
    ) AS invalid_date_attribute_count,

    SUM(
        CASE
            WHEN is_holiday IS NOT NULL THEN 1
            ELSE 0
        END
    ) AS unexpected_holiday_value_count

FROM `ftw-week-08`.`03_gold`.dim_date;



SELECT
    COUNT(*) AS affected_trip_rows,
    MIN(lpep_pickup_datetime) AS earliest_pickup_datetime,
    MAX(lpep_pickup_datetime) AS latest_pickup_datetime,
    MIN(lpep_dropoff_datetime) AS earliest_dropoff_datetime,
    MAX(lpep_dropoff_datetime) AS latest_dropoff_datetime
FROM `ftw-week-08`.`02_silver`.green_taxi
WHERE TO_DATE(lpep_pickup_datetime) = DATE '2008-12-31'
   OR TO_DATE(lpep_dropoff_datetime) = DATE '2008-12-31';

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.green_taxi
WHERE TO_DATE(lpep_pickup_datetime) = DATE '2008-12-31'
   OR TO_DATE(lpep_dropoff_datetime) = DATE '2008-12-31';

## Out-of-Range Date Investigation Result

The investigation found two retained Silver Green Taxi records with pickup dates on `2008-12-31`. One record also has a dropoff date on `2009-01-01`.

The records remain available in the Silver source, so the Gold `dim_date` dimension currently includes the related dates to preserve foreign-key coverage. A decision is needed from the Silver/data-quality owner on whether these records should be corrected or excluded upstream before the final Gold rerun.

## DIM_DATE Validation and Out-of-Range Date Investigation

The `dim_date` validation passed for completeness, uniqueness, continuity, and calendar-attribute accuracy.

The table contains 6,362 unique date records with no missing dates between the minimum and maximum values. The `is_holiday` column is intentionally `NULL` because no verified holiday source is currently available.

However, the minimum date is `2008-12-31`, which was previously flagged as an out-of-range datetime in the Silver layer. The next query investigates the affected Silver records before any filtering or removal decision is made.

In [0]:
SELECT *
FROM `ftw-week-08`.`03_gold`.dim_date
ORDER BY full_date
LIMIT 10;

In [0]:
DESCRIBE TABLE `ftw-week-08`.`03_gold`.dim_date;

## DIM_DATE Validation Result

The `dim_date` table was successfully created and validated.

The table contains 6,362 rows and 6,362 unique date keys. The date range covers 2008-12-31 to 2026-06-01.

The preview and schema inspection confirmed that the expected date attributes and data types are present. The dimension is ready to support the Gold fact table.

## Create DIM_TIME

The `dim_time` table contains one row for every hour of the day.

It provides reusable time attributes for pickup-hour and dropoff-hour analysis. Its expected grain is one row per hour.


In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_time
USING DELTA
AS
SELECT
    0 AS time_key,
    CAST(NULL AS INT) AS hour_24,
    'Unknown' AS hour_label,
    'Unknown' AS day_period

UNION ALL

SELECT
    hour_24 + 1 AS time_key,
    hour_24,
    CONCAT(
        LPAD(CAST(hour_24 AS STRING), 2, '0'),
        ':00'
    ) AS hour_label,
    CASE
        WHEN hour_24 < 12 THEN 'AM'
        ELSE 'PM'
    END AS day_period
FROM (
    SELECT EXPLODE(SEQUENCE(0, 23)) AS hour_24
);

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT time_key) AS unique_time_keys,
    MIN(time_key) AS minimum_time_key,
    MAX(time_key) AS maximum_time_key,

    SUM(
        CASE
            WHEN time_key = 0
             AND hour_24 IS NULL
             AND hour_label = 'Unknown'
             AND day_period = 'Unknown'
            THEN 0
            WHEN time_key = 0 THEN 1
            ELSE 0
        END
    ) AS invalid_unknown_row_count,

    SUM(
        CASE
            WHEN time_key BETWEEN 1 AND 24
             AND hour_24 = time_key - 1
            THEN 0
            WHEN time_key BETWEEN 1 AND 24 THEN 1
            ELSE 0
        END
    ) AS invalid_hour_mapping_count

FROM `ftw-week-08`.`03_gold`.dim_time;

## Gold Mart Preparation
The initial Gold mart preparation was completed using the approved star schema design.

The Silver-layer source tables were inspected and documented, including:

- `green_taxi`
- `taxi_zones`
- `weather`

The existing Gold tables were retained for reference and were not modified.

The new `dim_date` table was successfully created and validated with 6,362 rows and 6,362 unique date keys. The date range covers 2008-12-31 to 2026-06-01.

The `dim_time` table was also created with 24 unique hourly records, covering hours 00:00 to 23:00.

The next implementation steps are to create and validate:

- `dim_taxi_zone`
- `dim_weather_hour`
- `fact_green_taxi_trip`

The final fact table will follow the approved grain:

One row represents one Green Taxi trip.

The completed notebook and Gold-layer changes were saved and pushed to the feature branch for team review.

## Create DIM_TAXI_ZONE

The `dim_taxi_zone` table provides descriptive information about each taxi zone.

It will be used as a reusable dimension for both pickup and dropoff locations in the Green Taxi fact table.

Expected grain:

One row represents one taxi zone.

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.taxi_zones;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.taxi_zones
LIMIT 10;

## Taxi Zone Source Review

The taxi zone source table was inspected to confirm its available columns and structure.

The location identifier will be used as the primary key of `dim_taxi_zone`. The descriptive zone attributes will support pickup and dropoff location analysis.

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_taxi_zone
USING DELTA
AS
SELECT
    CAST(LocationID AS INT) AS taxi_zone_key,
    CAST(LocationID AS INT) AS location_id,
    Borough AS borough,
    Zone AS zone_name,
    service_zone
FROM `ftw-week-08`.`02_silver`.taxi_zones;

## Validate DIM_TAXI_ZONE

The `dim_taxi_zone` table will be validated by checking the total number of records, unique taxi zone keys, and null keys.

Each taxi zone should have one unique key, and the key should not be null.

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT taxi_zone_key) AS unique_taxi_zone_keys,

    COUNT(*) - COUNT(DISTINCT taxi_zone_key)
        AS duplicate_taxi_zone_key_count,

    SUM(
        CASE
            WHEN taxi_zone_key IS NULL THEN 1
            ELSE 0
        END
    ) AS null_taxi_zone_key_count,

    SUM(
        CASE
            WHEN location_id IS NULL THEN 1
            ELSE 0
        END
    ) AS null_location_id_count,

    SUM(
        CASE
            WHEN taxi_zone_key != location_id THEN 1
            ELSE 0
        END
    ) AS key_reference_mismatch_count

FROM `ftw-week-08`.`03_gold`.dim_taxi_zone;

## DIM_TAXI_ZONE Validation Result

The `dim_taxi_zone` table was created successfully from the Silver taxi zone source.

The validation confirms that the taxi zone keys are unique and that no null taxi zone keys are present.

The dimension is ready to support pickup-zone and dropoff-zone relationships in the Gold fact table.

## Create DIM_WEATHER_HOUR

The `dim_weather_hour` table stores one row for each hourly weather observation.

It will provide weather attributes such as temperature, precipitation, rain, snowfall, weather code, and wind speed.

The weather timestamp will be used as the unique hourly key for joining weather conditions to Green Taxi trips.

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.weather;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.weather
ORDER BY weather_datetime
LIMIT 10;

## Weather Source Review

The Silver weather table was inspected to confirm the timestamp and weather measurement columns.

The `weather_datetime` column represents the hourly observation time and will be used to create the weather dimension key.

The weather measurements will be retained as descriptive attributes for trip and weather analysis.

## Create DIM_WEATHER_HOUR

The `dim_weather_hour` table stores one record for each hourly weather observation.

The `weather_datetime` column is used as the unique weather-hour key. Weather measurements such as temperature, precipitation, rain, snowfall, weather code, and wind speed are retained for analysis.

Expected grain:

One row represents one hourly weather observation.

In [0]:
SHOW TABLES IN `ftw-week-08`.`03_gold`;

## DIM_WEATHER_HOUR Build and Unknown Weather Handling

The weather dimension retains the 2,208 validated hourly records from the Silver weather source.

One additional `Unknown` member uses `weather_hour_key = 0`. It represents trips whose pickup hour has no available weather observation.

This preserves all retained trip records and enables valid fact-to-weather foreign-key relationships without creating or assuming weather measurements.

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.dim_weather_hour
USING DELTA
AS
SELECT
    CAST(0 AS BIGINT) AS weather_hour_key,
    CAST(NULL AS TIMESTAMP) AS weather_timestamp_local,
    CAST(NULL AS DOUBLE) AS temperature_2m,
    CAST(NULL AS DOUBLE) AS precipitation,
    CAST(NULL AS DOUBLE) AS rain,
    CAST(NULL AS DOUBLE) AS snowfall,
    CAST(NULL AS BIGINT) AS weather_code,
    CAST(NULL AS DOUBLE) AS wind_speed_10m,
    'Unknown' AS timezone,
    CAST(NULL AS DOUBLE) AS latitude,
    CAST(NULL AS DOUBLE) AS longitude,
    CAST(NULL AS BIGINT) AS utc_offset_seconds,
    'Unknown' AS source_system,
    CAST(NULL AS STRING) AS source_file,
    CAST(NULL AS STRING) AS batch_id

UNION ALL

SELECT
    CAST(DATE_FORMAT(weather_datetime, 'yyyyMMddHH') AS BIGINT),
    weather_datetime,
    temperature_2m,
    precipitation,
    rain,
    snowfall,
    weather_code,
    wind_speed_10m,
    timezone,
    latitude,
    longitude,
    utc_offset_seconds,
    source_system,
    source_file,
    batch_id
FROM `ftw-week-08`.`02_silver`.weather;

In [0]:
WITH source_summary AS (
    SELECT
        COUNT(*) AS source_weather_row_count,
        COUNT(DISTINCT weather_datetime) AS source_unique_weather_hours
    FROM `ftw-week-08`.`02_silver`.weather
),
gold_actual_weather AS (
    SELECT
        COUNT(*) AS gold_actual_weather_row_count,
        COUNT(DISTINCT weather_hour_key) AS gold_actual_unique_weather_hour_keys
    FROM `ftw-week-08`.`03_gold`.dim_weather_hour
    WHERE weather_hour_key != 0
),
unknown_member AS (
    SELECT
        COUNT(*) AS unknown_weather_member_count
    FROM `ftw-week-08`.`03_gold`.dim_weather_hour
    WHERE weather_hour_key = 0
)
SELECT
    s.source_weather_row_count,
    s.source_unique_weather_hours,
    g.gold_actual_weather_row_count,
    g.gold_actual_unique_weather_hour_keys,
    g.gold_actual_weather_row_count - s.source_weather_row_count
        AS actual_weather_row_count_difference,
    u.unknown_weather_member_count
FROM source_summary AS s
CROSS JOIN gold_actual_weather AS g
CROSS JOIN unknown_member AS u;

## DIM_WEATHER_HOUR Validation Result

Validation confirmed that all 2,208 Silver weather-hour records were retained in Gold with unique weather-hour keys.

One additional `Unknown` weather member (`weather_hour_key = 0`) is intentionally present for retained trips whose pickup hour has no available Silver weather observation. This member contains no assumed weather measurements.

## Validate DIM_WEATHER_HOUR

The weather dimension will be validated by checking the total number of hourly records, unique weather timestamps, and null weather keys.

Each `weather_datetime` should represent one unique hourly observation.

In [0]:
WITH source_summary AS (
    SELECT
        COUNT(*) AS source_row_count,
        COUNT(DISTINCT weather_datetime) AS source_unique_weather_hours
    FROM `ftw-week-08`.`02_silver`.weather
),

gold_summary AS (
    SELECT
        COUNT(*) AS gold_row_count,
        COUNT(DISTINCT weather_hour_key) AS unique_weather_hour_keys,
        COUNT(DISTINCT weather_timestamp_local) AS unique_weather_timestamps,

        SUM(
            CASE
                WHEN weather_hour_key IS NULL THEN 1
                ELSE 0
            END
        ) AS null_weather_hour_key_count,

        SUM(
            CASE
                WHEN weather_timestamp_local IS NULL THEN 1
                ELSE 0
            END
        ) AS null_weather_timestamp_count,

        SUM(
            CASE
                WHEN weather_hour_key !=
                    CAST(
                        DATE_FORMAT(weather_timestamp_local, 'yyyyMMddHH')
                        AS BIGINT
                    )
                THEN 1
                ELSE 0
            END
        ) AS invalid_weather_key_mapping_count
    FROM `ftw-week-08`.`03_gold`.dim_weather_hour
)

SELECT
    s.source_row_count,
    s.source_unique_weather_hours,

    g.gold_row_count,
    g.unique_weather_hour_keys,
    g.unique_weather_timestamps,

    g.gold_row_count - s.source_row_count
        AS row_count_difference,

    g.null_weather_hour_key_count,
    g.null_weather_timestamp_count,
    g.invalid_weather_key_mapping_count

FROM source_summary AS s
CROSS JOIN gold_summary AS g;

In [0]:
SELECT *
FROM `ftw-week-08`.`03_gold`.dim_weather_hour
ORDER BY weather_datetime
LIMIT 10;

In [0]:
## DIM_WEATHER_HOUR Validation Result

The `dim_weather_hour` table was created directly from the validated Silver weather source.

Validation confirmed 2,208 source rows and 2,208 Gold rows. The weather-hour keys and local weather timestamps are unique, non-null, and correctly mapped.

No Gold-side deduplication was applied because the Silver layer already provides one approved record per weather hour. The dimension retains the weather attributes and source provenance needed for downstream analysis.

## Create FACT_GREEN_TAXI_TRIP

The `fact_green_taxi_trip` table stores the measurable business events from the Green Taxi trips.

Expected grain:

One row represents one Green Taxi trip.

The fact table will contain trip timestamps, pickup and dropoff keys, weather reference, trip measures, financial measures, and `batch_id` for lineage.

A deterministic `trip_key` will be generated because the source does not provide a guaranteed unique trip identifier.

In [0]:
DESCRIBE TABLE `ftw-week-08`.`02_silver`.green_taxi;

In [0]:
SELECT *
FROM `ftw-week-08`.`02_silver`.green_taxi
LIMIT 10;

## Create FACT_GREEN_TAXI_TRIP

The `fact_green_taxi_trip` table stores one record for each Green Taxi trip.

The deterministic `trip_key` is generated using stable trip attributes: VendorID, pickup and dropoff timestamps, pickup location, and dropoff location.

The fact table connects to the date, time, taxi zone, and weather dimensions through foreign keys.

Expected grain:

One row represents one Green Taxi trip.

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.fact_green_taxi_trip
USING DELTA
AS
SELECT
    xxhash64(
        g.VendorID,
        g.lpep_pickup_datetime,
        g.lpep_dropoff_datetime,
        g.store_and_fwd_flag,
        g.RatecodeID,
        g.PULocationID,
        g.DOLocationID,
        g.passenger_count,
        g.trip_distance,
        g.trip_duration_minutes,
        g.fare_amount,
        g.extra,
        g.mta_tax,
        g.tip_amount,
        g.tolls_amount,
        g.improvement_surcharge,
        g.total_amount,
        g.payment_type,
        g.trip_type,
        g.congestion_surcharge,
        g.cbd_congestion_fee,
        g.source_file
    ) AS trip_key,

    g.VendorID AS vendor_id,

    g.lpep_pickup_datetime AS pickup_datetime,
    g.lpep_dropoff_datetime AS dropoff_datetime,

    CAST(DATE_FORMAT(TO_DATE(g.lpep_pickup_datetime), 'yyyyMMdd') AS INT)
        AS pickup_date_key,
    CAST(DATE_FORMAT(TO_DATE(g.lpep_dropoff_datetime), 'yyyyMMdd') AS INT)
        AS dropoff_date_key,

    COALESCE(HOUR(g.lpep_pickup_datetime) + 1, 0)
        AS pickup_time_key,
    COALESCE(HOUR(g.lpep_dropoff_datetime) + 1, 0)
        AS dropoff_time_key,

    CAST(g.PULocationID AS INT) AS pickup_taxi_zone_key,
    CAST(g.DOLocationID AS INT) AS dropoff_taxi_zone_key,

    CAST(DATE_FORMAT(g.pickup_hour, 'yyyyMMddHH') AS BIGINT)
        AS pickup_weather_hour_key,

    CAST(1 AS BIGINT) AS trip_count,
    g.passenger_count,
    g.trip_distance,
    g.trip_duration_minutes,

    CASE
        WHEN g.trip_duration_minutes > 0
        THEN g.trip_distance / (g.trip_duration_minutes / 60.0)
    END AS average_speed_mph,

    g.fare_amount,
    g.extra,
    g.mta_tax,
    g.tip_amount,
    g.tolls_amount,
    g.improvement_surcharge,
    g.congestion_surcharge,
    g.cbd_congestion_fee,
    g.total_amount,

    g.payment_type,
    g.payment_type_description,
    g.trip_type,
    g.RatecodeID AS rate_code_id,
    g.store_and_fwd_flag,

    g.dq_zero_trip_distance,
    g.dq_extreme_trip_distance,
    g.dq_negative_trip_distance,
    g.dq_out_of_range_datetime,
    g.dq_invalid_trip_duration,

    g.source_system,
    g.source_file,
    g.batch_id

FROM `ftw-week-08`.`02_silver`.green_taxi AS g;

In [0]:
WITH silver_summary AS (
    SELECT
        COUNT(*) AS silver_row_count,
        SUM(fare_amount) AS silver_fare_amount,
        SUM(tip_amount) AS silver_tip_amount,
        SUM(total_amount) AS silver_total_amount
    FROM `ftw-week-08`.`02_silver`.green_taxi
),
gold_summary AS (
    SELECT
        COUNT(*) AS gold_row_count,
        COUNT(DISTINCT trip_key) AS unique_trip_keys,
        COUNT(*) - COUNT(DISTINCT trip_key) AS duplicate_trip_key_count,
        SUM(trip_count) AS gold_trip_count,
        SUM(fare_amount) AS gold_fare_amount,
        SUM(tip_amount) AS gold_tip_amount,
        SUM(total_amount) AS gold_total_amount
    FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip
),
foreign_key_validation AS (
    SELECT
        SUM(CASE WHEN pickup_date.date_key IS NULL THEN 1 ELSE 0 END)
            AS missing_pickup_date_key_count,
        SUM(CASE WHEN dropoff_date.date_key IS NULL THEN 1 ELSE 0 END)
            AS missing_dropoff_date_key_count,
        SUM(CASE WHEN pickup_time.time_key IS NULL THEN 1 ELSE 0 END)
            AS missing_pickup_time_key_count,
        SUM(CASE WHEN dropoff_time.time_key IS NULL THEN 1 ELSE 0 END)
            AS missing_dropoff_time_key_count,
        SUM(CASE WHEN pickup_zone.taxi_zone_key IS NULL THEN 1 ELSE 0 END)
            AS missing_pickup_zone_key_count,
        SUM(CASE WHEN dropoff_zone.taxi_zone_key IS NULL THEN 1 ELSE 0 END)
            AS missing_dropoff_zone_key_count,
        SUM(CASE WHEN weather.weather_hour_key IS NULL THEN 1 ELSE 0 END)
            AS missing_weather_hour_key_count
    FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f
    LEFT JOIN `ftw-week-08`.`03_gold`.dim_date AS pickup_date
        ON f.pickup_date_key = pickup_date.date_key
    LEFT JOIN `ftw-week-08`.`03_gold`.dim_date AS dropoff_date
        ON f.dropoff_date_key = dropoff_date.date_key
    LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS pickup_time
        ON f.pickup_time_key = pickup_time.time_key
    LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS dropoff_time
        ON f.dropoff_time_key = dropoff_time.time_key
    LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS pickup_zone
        ON f.pickup_taxi_zone_key = pickup_zone.taxi_zone_key
    LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS dropoff_zone
        ON f.dropoff_taxi_zone_key = dropoff_zone.taxi_zone_key
    LEFT JOIN `ftw-week-08`.`03_gold`.dim_weather_hour AS weather
        ON f.pickup_weather_hour_key = weather.weather_hour_key
)
SELECT
    s.silver_row_count,
    g.gold_row_count,
    g.gold_row_count - s.silver_row_count AS row_count_difference,
    g.unique_trip_keys,
    g.duplicate_trip_key_count,
    g.gold_trip_count,
    g.gold_fare_amount - s.silver_fare_amount AS fare_amount_difference,
    g.gold_tip_amount - s.silver_tip_amount AS tip_amount_difference,
    g.gold_total_amount - s.silver_total_amount AS total_amount_difference,
    f.*
FROM silver_summary AS s
CROSS JOIN gold_summary AS g
CROSS JOIN foreign_key_validation AS f;

## FACT_GREEN_TAXI_TRIP Validation Result

The Gold fact table retains all 133,367 Silver Green Taxi records.

Validation confirmed:

- 133,367 unique trip keys
- 0 duplicate trip keys
- 0 row-count difference between Silver and Gold
- 0 missing pickup and dropoff date keys
- 0 missing pickup and dropoff time keys
- 0 missing pickup and dropoff taxi-zone keys
- 0 missing pickup weather-hour keys

The small financial reconciliation differences are floating-point precision effects and are effectively zero. No records were removed through Gold-side deduplication.

In [0]:
SELECT
    f.pickup_weather_hour_key,
    f.pickup_datetime,
    f.dropoff_datetime,
    f.pickup_taxi_zone_key,
    f.dropoff_taxi_zone_key,
    f.source_file,
    f.dq_out_of_range_datetime
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f
LEFT JOIN `ftw-week-08`.`03_gold`.dim_weather_hour AS w
    ON f.pickup_weather_hour_key = w.weather_hour_key
WHERE w.weather_hour_key IS NULL
ORDER BY f.pickup_datetime;

## Validate FACT_GREEN_TAXI_TRIP Grain

The fact table will be validated according to its approved grain: one row represents one Green Taxi trip.

We will check the total number of records, unique trip keys, duplicate trip keys, and null trip keys.

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT trip_key) AS unique_trip_keys,
    COUNT(*) - COUNT(DISTINCT trip_key) AS duplicate_trip_count,
    SUM(
        CASE
            WHEN trip_key IS NULL THEN 1
            ELSE 0
        END
    ) AS null_trip_key_count
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip;

In [0]:
## Fact Trip Identity Investigation

The five-column combination is treated as a candidate business-key group only. It is not yet used as a final trip identifier or deduplication rule.

The next checks quantify the affected records and compare their measures before any record is removed from the Gold fact table.

In [0]:
WITH candidate_groups AS (
    SELECT
        VendorID,
        lpep_pickup_datetime,
        lpep_dropoff_datetime,
        PULocationID,
        DOLocationID,

        COUNT(*) AS candidate_group_rows,

        COUNT(DISTINCT trip_distance) AS distinct_trip_distances,
        COUNT(DISTINCT passenger_count) AS distinct_passenger_counts,
        COUNT(DISTINCT fare_amount) AS distinct_fare_amounts,
        COUNT(DISTINCT tip_amount) AS distinct_tip_amounts,
        COUNT(DISTINCT total_amount) AS distinct_total_amounts

    FROM `ftw-week-08`.`02_silver`.green_taxi

    GROUP BY
        VendorID,
        lpep_pickup_datetime,
        lpep_dropoff_datetime,
        PULocationID,
        DOLocationID

    HAVING COUNT(*) > 1
)

SELECT
    COUNT(*) AS candidate_group_count,
    SUM(candidate_group_rows) AS candidate_row_count,
    SUM(candidate_group_rows - 1) AS potential_rows_removed_if_deduplicated,

    SUM(
        CASE
            WHEN distinct_trip_distances > 1
              OR distinct_passenger_counts > 1
              OR distinct_fare_amounts > 1
              OR distinct_tip_amounts > 1
              OR distinct_total_amounts > 1
            THEN 1
            ELSE 0
        END
    ) AS groups_with_different_measures

FROM candidate_groups;

## Fact Trip Identity Investigation Result

The investigation identified 384 five-column candidate groups involving 768 Silver records.

Deduplicating one row per group would remove 384 records. All 384 groups contain differing business measures, so the five-column combination is not a safe final trip-identity or deduplication rule.

No candidate rows will be removed until an approved stable Silver record identifier or deterministic identity rule is confirmed.

In [0]:
WITH proposed_identity AS (
    SELECT
        xxhash64(
            VendorID,
            lpep_pickup_datetime,
            lpep_dropoff_datetime,
            store_and_fwd_flag,
            RatecodeID,
            PULocationID,
            DOLocationID,
            passenger_count,
            trip_distance,
            trip_duration_minutes,
            fare_amount,
            extra,
            mta_tax,
            tip_amount,
            tolls_amount,
            improvement_surcharge,
            total_amount,
            payment_type,
            trip_type,
            congestion_surcharge,
            cbd_congestion_fee,
            source_file
        ) AS proposed_record_key
    FROM `ftw-week-08`.`02_silver`.green_taxi
)

SELECT
    COUNT(*) AS silver_row_count,
    COUNT(DISTINCT proposed_record_key) AS unique_proposed_record_keys,
    COUNT(*) - COUNT(DISTINCT proposed_record_key)
        AS exact_duplicate_record_count,
    COUNT_IF(proposed_record_key IS NULL)
        AS null_proposed_record_key_count
FROM proposed_identity;

## Proposed Deterministic Record-Key Result

The full-record identity check returned 133,367 Silver rows and 133,367 unique proposed record keys, with zero exact duplicate records and zero null keys.

The proposed deterministic key uses stable trip and financial attributes plus `source_file`. It intentionally excludes run metadata and data-quality flags so the key remains stable across reruns.

This rule retains all Silver records and avoids the unsafe five-column `ROW_NUMBER()` deduplication approach.

In [0]:
CREATE OR REPLACE TABLE `ftw-week-08`.`03_gold`.fact_green_taxi_trip
USING DELTA
AS
SELECT
    xxhash64(
        g.VendorID,
        g.lpep_pickup_datetime,
        g.lpep_dropoff_datetime,
        g.store_and_fwd_flag,
        g.RatecodeID,
        g.PULocationID,
        g.DOLocationID,
        g.passenger_count,
        g.trip_distance,
        g.trip_duration_minutes,
        g.fare_amount,
        g.extra,
        g.mta_tax,
        g.tip_amount,
        g.tolls_amount,
        g.improvement_surcharge,
        g.total_amount,
        g.payment_type,
        g.trip_type,
        g.congestion_surcharge,
        g.cbd_congestion_fee,
        g.source_file
    ) AS trip_key,

    g.VendorID AS vendor_id,
    g.lpep_pickup_datetime AS pickup_datetime,
    g.lpep_dropoff_datetime AS dropoff_datetime,

    CAST(DATE_FORMAT(TO_DATE(g.lpep_pickup_datetime), 'yyyyMMdd') AS INT)
        AS pickup_date_key,

    CAST(DATE_FORMAT(TO_DATE(g.lpep_dropoff_datetime), 'yyyyMMdd') AS INT)
        AS dropoff_date_key,

    COALESCE(HOUR(g.lpep_pickup_datetime) + 1, 0)
        AS pickup_time_key,

    COALESCE(HOUR(g.lpep_dropoff_datetime) + 1, 0)
        AS dropoff_time_key,

    CAST(g.PULocationID AS INT) AS pickup_taxi_zone_key,
    CAST(g.DOLocationID AS INT) AS dropoff_taxi_zone_key,

    CASE
        WHEN g.dq_out_of_range_datetime THEN 0
        ELSE CAST(DATE_FORMAT(g.pickup_hour, 'yyyyMMddHH') AS BIGINT)
    END AS pickup_weather_hour_key,

    CAST(1 AS BIGINT) AS trip_count,

    g.passenger_count,
    g.trip_distance,
    g.trip_duration_minutes,

    CASE
        WHEN g.trip_duration_minutes > 0
        THEN g.trip_distance / (g.trip_duration_minutes / 60.0)
    END AS average_speed_mph,

    g.fare_amount,
    g.extra,
    g.mta_tax,
    g.tip_amount,
    g.tolls_amount,
    g.improvement_surcharge,
    g.congestion_surcharge,
    g.cbd_congestion_fee,
    g.total_amount,

    g.payment_type,
    g.payment_type_description,
    g.trip_type,
    g.RatecodeID AS rate_code_id,
    g.store_and_fwd_flag,

    g.dq_zero_trip_distance,
    g.dq_extreme_trip_distance,
    g.dq_negative_trip_distance,
    g.dq_out_of_range_datetime,
    g.dq_invalid_trip_duration,

    g.dq_out_of_range_datetime AS dq_missing_weather_coverage,

    g.source_system,
    g.source_file,
    g.batch_id

FROM `ftw-week-08`.`02_silver`.green_taxi AS g;

In [0]:
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT trip_key) AS unique_trip_keys,
    COUNT(*) - COUNT(DISTINCT trip_key) AS duplicate_trip_count,
    SUM(
        CASE
            WHEN trip_key IS NULL THEN 1
            ELSE 0
        END
    ) AS null_trip_key_count
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip;

In [0]:
SELECT
    COUNT(*) AS fact_row_count,
    COUNT(DISTINCT f.trip_key) AS fact_unique_trip_keys,
    COUNT(*) - COUNT(DISTINCT f.trip_key)
        AS duplicate_rows_after_dimension_joins
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f
LEFT JOIN `ftw-week-08`.`03_gold`.dim_date AS pickup_date
    ON f.pickup_date_key = pickup_date.date_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_date AS dropoff_date
    ON f.dropoff_date_key = dropoff_date.date_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS pickup_time
    ON f.pickup_time_key = pickup_time.time_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS dropoff_time
    ON f.dropoff_time_key = dropoff_time.time_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS pickup_zone
    ON f.pickup_taxi_zone_key = pickup_zone.taxi_zone_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS dropoff_zone
    ON f.dropoff_taxi_zone_key = dropoff_zone.taxi_zone_key
LEFT JOIN `ftw-week-08`.`03_gold`.dim_weather_hour AS weather
    ON f.pickup_weather_hour_key = weather.weather_hour_key;

## Fact-to-Dimension Join Cardinality Validation

The fact table was joined to the date, time, taxi-zone, and weather dimensions using their respective surrogate keys.

The joined result retained 133,367 rows and 133,367 unique trip keys. No duplicate rows were introduced through the dimension joins.

## Validate FACT Foreign-Key Relationships

The fact table will be checked against the Gold dimensions to confirm that every foreign key has a matching dimension record.

This validates the relationships for:

- Pickup and dropoff dates
- Pickup and dropoff times
- Pickup and dropoff taxi zones
- Pickup weather hour

Unmatched keys may indicate missing dimension records or incorrect source values.

In [0]:
SELECT
    SUM(CASE WHEN d_pickup.date_key IS NULL THEN 1 ELSE 0 END)
        AS missing_pickup_date_keys,

    SUM(CASE WHEN d_dropoff.date_key IS NULL THEN 1 ELSE 0 END)
        AS missing_dropoff_date_keys,

    SUM(CASE WHEN t_pickup.time_key IS NULL THEN 1 ELSE 0 END)
        AS missing_pickup_time_keys,

    SUM(CASE WHEN t_dropoff.time_key IS NULL THEN 1 ELSE 0 END)
        AS missing_dropoff_time_keys,

    SUM(CASE WHEN z_pickup.taxi_zone_key IS NULL THEN 1 ELSE 0 END)
        AS missing_pickup_zone_keys,

    SUM(CASE WHEN z_dropoff.taxi_zone_key IS NULL THEN 1 ELSE 0 END)
        AS missing_dropoff_zone_keys,

    SUM(CASE WHEN w.weather_datetime IS NULL THEN 1 ELSE 0 END)
        AS missing_weather_keys

FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f

LEFT JOIN `ftw-week-08`.`03_gold`.dim_date AS d_pickup
    ON f.pickup_date_key = d_pickup.date_key

LEFT JOIN `ftw-week-08`.`03_gold`.dim_date AS d_dropoff
    ON f.dropoff_date_key = d_dropoff.date_key

LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS t_pickup
    ON f.pickup_time_key = t_pickup.time_key

LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS t_dropoff
    ON f.dropoff_time_key = t_dropoff.time_key

LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS z_pickup
    ON f.pickup_taxi_zone_key = z_pickup.taxi_zone_key

LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS z_dropoff
    ON f.dropoff_taxi_zone_key = z_dropoff.taxi_zone_key

LEFT JOIN `ftw-week-08`.`03_gold`.dim_weather_hour AS w
    ON f.weather_datetime = w.weather_datetime;

## Foreign-Key Validation Result

The fact table was checked against all Gold dimensions.

A result of zero for all missing-key checks confirms that the fact table can successfully connect to the date, time, taxi zone, and weather dimensions.

## Reconcile Gold Fact with Silver Source

The Gold fact table will be reconciled with the Silver Green Taxi source.

We will compare the number of unique trips and the total revenue between both layers.

This confirms that the Gold transformation preserved the expected business records and financial totals after removing duplicate trip records.

In [0]:
WITH silver_trip_summary AS (
    SELECT
        COUNT(DISTINCT
            xxhash64(
                CAST(VendorID AS STRING),
                CAST(lpep_pickup_datetime AS STRING),
                CAST(lpep_dropoff_datetime AS STRING),
                CAST(PULocationID AS STRING),
                CAST(DOLocationID AS STRING)
            )
        ) AS silver_unique_trips,
        SUM(total_amount) AS silver_total_revenue
    FROM `ftw-week-08`.`02_silver`.green_taxi
),

gold_trip_summary AS (
    SELECT
        COUNT(DISTINCT trip_key) AS gold_unique_trips,
        SUM(total_amount) AS gold_total_revenue
    FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip
)

SELECT
    silver_unique_trips,
    gold_unique_trips,
    silver_unique_trips - gold_unique_trips AS trip_count_difference,

    silver_total_revenue,
    gold_total_revenue,
    silver_total_revenue - gold_total_revenue AS revenue_difference
FROM silver_trip_summary
CROSS JOIN gold_trip_summary;

## Reconciliation Result

The Gold fact table was compared with the Silver Green Taxi source using the approved deterministic trip key.

The reconciliation documents the trip-count and revenue differences caused by removing duplicate trip records.

The Gold layer prioritizes a reliable one-row-per-trip grain, unique trip keys, valid dimension relationships, and traceable lineage through `batch_id`.

## Business Question 1: Revenue by Taxi Zone

This query identifies the pickup taxi zones that generate the highest total revenue.

It uses the fact table for financial measures and `dim_taxi_zone` for descriptive zone names.

In [0]:
SELECT
    z.zone_name AS pickup_zone,
    z.borough,
    COUNT(*) AS total_trips,
    ROUND(SUM(f.total_amount), 2) AS total_revenue
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f
LEFT JOIN `ftw-week-08`.`03_gold`.dim_taxi_zone AS z
    ON f.pickup_taxi_zone_key = z.taxi_zone_key
GROUP BY
    z.zone_name,
    z.borough
ORDER BY total_revenue DESC
LIMIT 10;

## Business Question 1: Which pickup zones generate the most trips and revenue?

**Answer:**  
East Harlem North in Manhattan ranks first in the result, with 34,826 trips and the highest total revenue among the listed pickup zones.

**Meaning:**  
This indicates that East Harlem North is a major pickup area and an important revenue-generating zone. The top 10 results can support demand planning, zone prioritization, and operational decisions.

The analysis uses `fact_green_taxi_trip` joined to `dim_taxi_zone` through `pickup_taxi_zone_key`.

## Business Question 2: Trip Demand by Hour

This query shows the number of Green Taxi trips by pickup hour.

It uses `dim_time` to provide readable time labels for hourly demand analysis.

In [0]:
SELECT
    t.time_key AS pickup_hour,
    t.hour_label,
    t.day_period,
    COUNT(*) AS total_trips,
    ROUND(SUM(f.total_amount), 2) AS total_revenue
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f
LEFT JOIN `ftw-week-08`.`03_gold`.dim_time AS t
    ON f.pickup_time_key = t.time_key
GROUP BY
    t.time_key,
    t.hour_label,
    t.day_period
ORDER BY pickup_hour;

## Business Question 2: How do trip activity and revenue vary by pickup hour?

**Answer:**  
Trip activity changes throughout the day. The results show lower demand during the early morning, increasing activity from the morning onward, and higher demand during the daytime and afternoon hours.

**Meaning:**  
Pickup demand is time-dependent. This hourly pattern can help the team identify high-demand periods for operational planning and resource allocation.

The analysis uses `fact_green_taxi_trip` joined to `dim_time` through `pickup_time_key`, following the convention `1–24 = real hours`.

## Business Question 3: Trips by Weather Condition

This query examines trip activity alongside hourly weather conditions.

It combines the Green Taxi fact table with `dim_weather_hour` using the pickup weather hour.

In [0]:
SELECT
    w.weather_code,
    COUNT(*) AS total_trips,
    ROUND(AVG(w.temperature_2m), 2) AS average_temperature,
    ROUND(SUM(f.total_amount), 2) AS total_revenue
FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip AS f
LEFT JOIN `ftw-week-08`.`03_gold`.dim_weather_hour AS w
    ON f.pickup_weather_hour_key = w.weather_hour_key
GROUP BY w.weather_code
ORDER BY total_trips DESC;

## Business Question 3: How do trips and revenue vary by weather condition?

**Answer:**  
Weather code `0` has the highest trip volume, with 50,270 trips and an average temperature of 13.66. Other weather codes also contribute to trip activity and revenue.

**Meaning:**  
The results show how trips and revenue are distributed across observed weather conditions. The `NULL` weather-code group represents the 19 retained trips with unavailable weather observations and should not be interpreted as a measured weather condition.

The analysis uses `fact_green_taxi_trip` joined to `dim_weather_hour` through `pickup_weather_hour_key`.

## Business Questions Result Summary

The three queries successfully use the final Gold star schema:

- Pickup-zone analysis uses `dim_taxi_zone`.
- Hourly trip analysis uses `dim_time`.
- Weather analysis uses `dim_weather_hour`.

The results are based on the final validated `fact_green_taxi_trip` table, which retains all 133,367 Silver records without Gold-side deduplication.

## Final Gold Mart Validation Conclusion

The Gold mart was rebuilt from the approved Silver sources.

The final fact table retains all 133,367 Silver Green Taxi records with unique deterministic trip keys. Date, time, taxi-zone, and weather foreign-key validations passed with zero missing references. Dimension joins also preserved the fact-table row count without multiplication.

The unsafe candidate-group deduplication was removed. All retained records remain traceable through the source and data-quality columns.